# T3b — Normalización de la tarifa y tamaño de familia
**Autor/a:** Sergio Rodríguez
**Proyecto P1:** Análisis exploratorio del Titanic (numpy y pandas)

## Objetivo

Crear dos columnas nuevas con operaciones vectorizadas, sin bucles:
- `tam_familia`: número de familiares a bordo más el propio pasajero (`sibsp + parch + 1`).
- `tarifa_norm`: la tarifa normalizada entre 0 y 1 (normalización min-max), para poder compararla en una escala común.

In [1]:
import pandas as pd
import numpy as np

# Carga del dataset desde la carpeta Data del repositorio.
# Si no se encuentra (por ejemplo al abrir el notebook en Colab), se descarga
# de la misma fuente original, así que los datos son idénticos.
try:
    df = pd.read_csv("../Data/titanic.csv")
except FileNotFoundError:
    df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv")

df.shape

(891, 15)

Aplicamos la base común del contrato (tratamiento de nulos, ver T2a).

In [2]:
# --- BASE COMÚN (tratamiento de nulos acordado en el contrato, ver T2a) ---
df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df["embark_town"] = df["embark_town"].fillna(df["embark_town"].mode()[0])
df = df.drop(columns="deck")

In [3]:
# Tamaño de la familia: hermanos/cónyuges + padres/hijos + el propio pasajero
df["tam_familia"] = df["sibsp"] + df["parch"] + 1

# Normalización min-max de la tarifa: (valor - mínimo) / (máximo - mínimo)
# La operación se aplica a toda la columna a la vez (vectorizada)
fare_min = df["fare"].min()
fare_max = df["fare"].max()
df["tarifa_norm"] = (df["fare"] - fare_min) / (fare_max - fare_min)

df[["tam_familia", "tarifa_norm"]].describe()

,tam_familia,tarifa_norm
count,891.000000,891.000000
mean,1.904602,0.062858
std,1.613459,0.096995
min,1.000000,0.000000
25%,1.000000,0.015440
50%,1.000000,0.028213
75%,2.000000,0.060508
max,11.000000,1.000000


In [4]:
# Comprobamos que la normalización es correcta: el mínimo debe ser 0 y el máximo 1
assert np.isclose(df["tarifa_norm"].min(), 0)
assert np.isclose(df["tarifa_norm"].max(), 1)

# Cuántos pasajeros hay de cada tamaño de familia
print(df["tam_familia"].value_counts().sort_index())

# Relación entre tamaño de familia y tarifa pagada
print(df[["tarifa_norm", "tam_familia"]].corr().round(3))

# Tarifa normalizada media en cada clase
print(df.groupby("pclass")["tarifa_norm"].mean().round(3))

tam_familia
1     537
2     161
3     102
4      29
5      15
6      22
7      12
8       6
11      7
Name: count, dtype: int64
             tarifa_norm  tam_familia
tarifa_norm        1.000        0.217
tam_familia        0.217        1.000
pclass
1    0.164
2    0.040
3    0.027
Name: tarifa_norm, dtype: float64


## Hallazgo

El 60% de los pasajeros (537 de 891) viajaba solo. La tarifa normalizada media en 1ª clase (0,164) fue más de 6 veces la de 3ª clase (0,027), y la relación entre tamaño de familia y tarifa pagada es débil (correlación de 0,217).